# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedumer1941/Flyrank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring**

This notebook trains and evaluates classification models to predict which pages are
declining and need content refresh. The baseline heuristic score (from Week 4) is
our reference point — models must beat it on Precision@50 and ROC-AUC.

> **Label**: `is_declining = (trend_direction == "down")` — 54.2% of pages are declining
> **Data**: 30,000 pseudonymized pages, 32 clients
> **Split**: Client-holdout (20% of clients held out as test)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose **two models** to compare:

1. **Logistic Regression** — transparent, interpretable coefficients. Lane 2 needs a
   model the content team can explain to stakeholders. Logistic regression gives us
   feature weights (positive = increases decline probability) and a probabilistic score.

2. **Random Forest Classifier** — captures non-linear interactions and heavy-tailed
   distributions without manual feature engineering. The impression and click fields have
   extreme heavy tails (max impressions = 517K, median = 731), so a tree-based model
   that handles non-linearities naturally is a strong candidate.

**Metric**: Precision@50 (does the top-50 queue actually contain declining pages?),
Precision@100, and ROC-AUC. The content team has limited capacity to review pages,
so precision in the top of the queue matters more than overall accuracy.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# --- Load data ---
URL = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# --- Build features ---
numeric_features = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d',
    'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X_num = df[numeric_features].copy()

# Impute ALL numeric features to satisfy LogisticRegression
for col in numeric_features:
    if col in ['search_volume', 'competition', 'cpc', 'word_count', 'char_count']:
        X_num[col] = df.groupby('content_type')[col].transform(lambda s: s.fillna(s.median()))
    X_num[col] = X_num[col].fillna(X_num[col].median())

X_num.loc[(X_num['avg_position'] == 0), 'avg_position'] = X_num['avg_position'].median()

# Engineered features
X_num['ctr_gap'] = (X_num['ctr'].max() - X_num['ctr']) / (X_num['ctr'].max() - X_num['ctr'].min() + 1e-6)
X_num['imp_to_session'] = X_num['sessions_90d'] / (X_num['impressions_90d'] + 1e-6)
X_num['staleness_w'] = X_num['days_since_last_update'] / (X_num['content_age_days'] + 1e-6)
X_num['eng_per_session'] = X_num['engaged_sessions_90d'] / (X_num['sessions_90d'] + 1e-6)
X_num['ai_ratio'] = X_num['ai_sessions_90d'] / (X_num['sessions_90d'] + 1e-6)

# Categorical features
cat_feats = ['content_type', 'main_intent', 'competition_level',
             'age_tier', 'freshness_tier', 'impression_tier',
             'word_count_tier', 'position_tier']
X_cat = pd.get_dummies(df[cat_feats], drop_first=True)

# Final feature matrix
X = pd.concat([X_num, X_cat], axis=1).values.astype(float)
y = df['is_declining'].values
clients = df['client_id'].values
feature_names = list(X_num.columns) + list(X_cat.columns)

# --- Client-holdout split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clients))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Preprocessing complete. All NaNs handled.")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Preprocessing complete. All NaNs handled.
Train shape: (23837, 50), Test shape: (6163, 50)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The split is **client-holdout** using `GroupShuffleSplit`:
- 20% of **clients** (not rows) are held out as the test set
- No pages from a test client appear in training
- This simulates the real-world scenario where we recommend refresh actions for a
  *new* client whose data we haven't seen before

**Why not time-aware?** The dataset is a single cross-sectional snapshot (trailing-90d
metrics at export time), so there's no temporal ordering to exploit. A client-holdout
split is the right honesty check for cross-sectional content scoring.

In [10]:
# Verification of Section 2: Split Design
train_clients_list = set(clients[train_idx])
test_clients_list = set(clients[test_idx])

# Intersection should be empty
leakage = train_clients_list.intersection(test_clients_list)

print("=== SPLIT DESIGN VERIFICATION ===")
print(f"Total Unique Clients: {len(set(clients))}")
print(f"Train Clients: {len(train_clients_list)}")
print(f"Test Clients: {len(test_clients_list)}")
print(f"Client Leakage Detected: {len(leakage)}")
print(f"Train set percentage: {len(train_idx)/len(df)*100:.1f}%")
print(f"Test set percentage: {len(test_idx)/len(df)*100:.1f}%")

=== SPLIT DESIGN VERIFICATION ===
Total Unique Clients: 32
Train Clients: 25
Test Clients: 7
Client Leakage Detected: 0
Train set percentage: 79.5%
Test set percentage: 20.5%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The baseline from Week 4 is a hand-crafted heuristic score:
```
baseline_score = (imp_norm * 0.4) + (ctr_gap * 0.3) + (staleness_norm * 0.3)
```
Both models and the baseline are evaluated on the **same test set**.

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Build baseline heuristic score (same as Week 4) ---
imp_col = 'impressions_90d'
stale_col = 'days_since_last_update'
ctr_col = 'ctr'

df_test = df.iloc[test_idx]
imp_norm = (df_test[imp_col] - df_test[imp_col].min()) / (df_test[imp_col].max() - df_test[imp_col].min() + 1e-6)
stale_norm = (df_test[stale_col] - df_test[stale_col].min()) / (df_test[stale_col].max() - df_test[stale_col].min() + 1e-6)
ctr_gap_baseline = (df_test[ctr_col].max() - df_test[ctr_col]) / (df_test[ctr_col].max() - df_test[ctr_col].min() + 1e-6)
baseline_scores = (imp_norm * 0.4) + (ctr_gap_baseline * 0.3) + (stale_norm * 0.3)

# --- Train Logistic Regression ---
print("Training Logistic Regression...", end=" ")
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
print("done.")

# --- Train Random Forest ---
print("Training Random Forest (100 trees)...", end=" ")
rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                             random_state=42, class_weight='balanced',
                             n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
print("done.\n")

# --- Evaluate at K ---
def precision_at_k(y_true, scores, k):
    top_k = np.argsort(scores)[-k:][::-1]
    return y_true[top_k].mean()

results = []
for name, scores in [('Baseline', baseline_scores.values),
                      ('Logistic Reg', lr_probs),
                      ('Random Forest', rf_probs)]:
    p50 = precision_at_k(y_test, scores, 50)
    p100 = precision_at_k(y_test, scores, 100)
    p200 = precision_at_k(y_test, scores, 200)
    auc = roc_auc_score(y_test, scores)
    results.append((name, p50, p100, p200, auc))

print("=== MODEL COMPARISON ===\n")
print(f"{'':20s} {'Precision@50':>14s} {'Precision@100':>15s} {'Precision@200':>15s} {'ROC-AUC':>8s}")
for name, p50, p100, p200, auc in results:
    print(f"{name:20s} {p50:>14.3f} {p100:>15.3f} {p200:>15.3f} {auc:>8.3f}")

print(f"\nBest model by ROC-AUC: Random Forest ({results[2][4]:.3f})")
print(f"Lift over baseline: +{results[2][4]-results[0][4]:.3f} (+{(results[2][4]/results[0][4]-1)*100:.1f}%)\n")

# --- Feature importance ---
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]
print("=== RANDOM FOREST: Top 10 Features ===")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {feature_names[idx]:30s} ({importances[idx]:.3f})")

Training Logistic Regression... done.
Training Random Forest (100 trees)... done.

=== MODEL COMPARISON ===

                       Precision@50   Precision@100   Precision@200  ROC-AUC
Baseline                      0.320           0.400           0.430    0.454
Logistic Reg                  0.760           0.680           0.695    0.604
Random Forest                 0.600           0.580           0.640    0.620

Best model by ROC-AUC: Random Forest (0.620)
Lift over baseline: +0.166 (+36.6%)

=== RANDOM FOREST: Top 10 Features ===
   1. impressions_90d                (0.100)
   2. days_with_impressions          (0.097)
   3. staleness_w                    (0.076)
   4. content_age_days               (0.072)
   5. imp_to_session                 (0.071)
   6. avg_position                   (0.061)
   7. char_count                     (0.045)
   8. word_count                     (0.042)
   9. scroll_rate                    (0.035)
  10. ctr                            (0.027)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
from sklearn.metrics import classification_report

# Convert probabilities to binary predictions
rf_preds = (rf_probs >= 0.5).astype(int)

print("=== ERROR ANALYSIS (Random Forest, Test Set) ===\n")
print("Classification Report (threshold=0.5):")
print(classification_report(y_test, rf_preds,
      target_names=['Not Declining', 'Declining']))

# Analysis logic
print("\n--- Insights ---")
print("1. Generalization: The model successfully generalizes to new clients (test set) using the client-holdout split.")
print("2. Key Signals: Top predictors include 'impressions_90d' and 'staleness_w', confirming that traffic scale and decay are dominant factors.")
print("3. Improvement: The significant lift in Precision@50 (0.60 vs 0.32) suggests the model is much better at prioritizing the content refresh queue than the manual heuristic.")

=== ERROR ANALYSIS (Random Forest, Test Set) ===

Classification Report (threshold=0.5):
               precision    recall  f1-score   support

Not Declining       0.58      0.55      0.57      3014
    Declining       0.59      0.62      0.61      3149

     accuracy                           0.59      6163
    macro avg       0.59      0.59      0.59      6163
 weighted avg       0.59      0.59      0.59      6163


--- Insights ---
1. Generalization: The model successfully generalizes to new clients (test set) using the client-holdout split.
2. Key Signals: Top predictors include 'impressions_90d' and 'staleness_w', confirming that traffic scale and decay are dominant factors.
3. Improvement: The significant lift in Precision@50 (0.60 vs 0.32) suggests the model is much better at prioritizing the content refresh queue than the manual heuristic.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.